# 📦 Step 1 — 下载「去审查版」Qwen3.6-35B-A3B 并打包成 Kaggle Dataset

这个 notebook 只跑 **一次**：把**已消融 / 去审查**的 Qwen3.6-35B-A3B GGUF 从 HuggingFace
直接下到 Kaggle 服务器上，再打包成一个**永久私有 Kaggle Dataset**。

之后 `2_run_server.ipynb` 每次启动都能秒挂载，不用重复下载。

---

### 和原版仓库的区别

| 项 | 原版 | 这一版 |
|---|---|---|
| 权重 | `unsloth/Qwen3.6-35B-A3B-GGUF`（官方对齐版，强拒答） | `llmfan46/...-uncensored-heretic-GGUF`（heretic 消融，几乎不拒答） |
| 量化 | `UD-Q4_K_XL` | `Q4_K_M`（体积接近，显存占用更友好） |
| 视觉 | `mmproj-F16.gguf` | 同仓库 `mmproj-BF16.gguf`（同样 0.84 GB，图生照旧可用） |
| Dataset slug | `qwen36-35b-a3b-gguf` | `qwen36-35b-a3b-uncensored-gguf` |
| 文件名 | 写死在 notebook 里 | 自动写 `manifest.json`，服务端自动读取 → 换预设不用改第二个 notebook |

---

### 三个预设（改 Cell 2 的 `PRESET` 一行即可切换）

| 预设 | 仓库 | 量化 | 体积 | 特点 |
|---|---|---|---|---|
| **A（默认，推荐）** | `llmfan46/Qwen3.6-35B-A3B-uncensored-heretic-GGUF` | `Q4_K_M` | 19.8 GB | heretic 用 KL 散度约束做消融，**能力损失最小**，拒答基本消失 |
| **B（最彻底）** | `Unrestricted/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive` | `Q4_K_P` | 21.8 GB | 去审查微调，最贴近“完全不拒答”，风格更放飞 |
| **C（经典）** | `huihui-ai/Huihui-Qwen3.6-35B-A3B-abliterated-MTP-GGUF` | `Q4_K` | 20.2 GB | 老牌 abliteration（方向正交消融），多语言最稳 |

> 三个都是 **非 gated** 仓库，不需要 HF token，也不需要登录。
> 三个都自带 `mmproj`（视觉投影），所以**多模态能力不会因为去审查而丢掉**。

---

### 前置条件

1. 右侧 Notebook Settings → **Internet → ON**
2. Accelerator → **None**（这一步只用 CPU）
3. Kaggle 免费额度 100 GB，够放


## ⚙️ Cell 1 — 装依赖

- `huggingface_hub` — 官方下载库
- `hf_transfer` — Rust 传输后端，大文件快 3–5 倍


In [ ]:
%%capture
!pip install -q huggingface_hub hf_transfer


## ⚙️ Cell 2 — 配置（唯一需要动的地方）

`PRESET` 就是模型选择：`"A"` / `"B"` / `"C"`。

下载目标是 `/tmp/model/`，不要改到 `/kaggle/working/`——那里只有 20 GB 配额，
装不下一个近 20 GB 的 gguf 再加一份上传副本。`/tmp` 吃的是整块 ~73 GB 磁盘。


In [ ]:
import os
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # Rust 传输后端，大文件快 3–5 倍

# ══════════════════════════════════════════════════════════════════════════════
# 选模型：改这一个字母
#   A = heretic 消融（推荐，能力保留最好）
#   B = Aggressive 去审查微调（最不拒答）
#   C = 经典 abliteration（huihui-ai，多语言稳）
PRESET = "A"
# ══════════════════════════════════════════════════════════════════════════════

PRESETS = {
    "A": dict(
        repo  = "llmfan46/Qwen3.6-35B-A3B-uncensored-heretic-GGUF",
        quant = "Q4_K_M",
        note  = "heretic 消融 · 能力保留最好 (19.8 GB)",
    ),
    "B": dict(
        repo  = "Unrestricted/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive",
        quant = "Q4_K_P",
        note  = "Aggressive 去审查微调 · 最不拒答 (21.8 GB)",
    ),
    "C": dict(
        repo  = "huihui-ai/Huihui-Qwen3.6-35B-A3B-abliterated-MTP-GGUF",
        quant = "Q4_K",
        note  = "经典 abliteration · 多语言最稳 (20.2 GB)",
    ),
}
assert PRESET in PRESETS, f"PRESET 只能是 {list(PRESETS)}"

REPO_ID = PRESETS[PRESET]["repo"]
QUANT   = PRESETS[PRESET]["quant"]

DOWNLOAD_DIR = Path("/tmp/model")

# Dataset 名称（只能小写 + 连字符；改了这里，2_run_server 的 DATASET_NAME 也要跟着改）
DATASET_SLUG  = "qwen36-35b-a3b-uncensored-gguf"
DATASET_TITLE = "Qwen3.6-35B-A3B Uncensored GGUF"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print(f"🎯 预设       : {PRESET} — {PRESETS[PRESET]['note']}")
print(f"🤗 源仓库     : {REPO_ID}")
print(f"🗜️  量化       : {QUANT}")
print(f"📁 下载目录   : {DOWNLOAD_DIR}")
print(f"📦 Dataset    : {DATASET_SLUG}")


## 📥 Cell 3 — 下载

只拉两个文件：主模型（`*<QUANT>*.gguf`）+ 视觉投影（`*mmproj*.gguf`）。
不会把仓库里十几个量化版本全拖下来。

> ⏳ 15–30 分钟（Kaggle ↔ HuggingFace 直连，不经过你本机）


In [ ]:
from huggingface_hub import snapshot_download

print(f"🚀 开始下载 {REPO_ID} [{QUANT}] ...")
print("   预计 15–30 分钟，进度条刷在下面。\n")

snapshot_download(
    repo_id        = REPO_ID,
    repo_type      = "model",
    local_dir      = str(DOWNLOAD_DIR),
    allow_patterns = [f"*<QUANT>*.gguf", "*mmproj*.gguf"],
)

print("\n✅ 下载完成。")


## 🔍 Cell 4 — 校验 + 写 manifest

三件事：

1. 文件存在、体积合理（主模型 > 15 GB，mmproj ≈ 0.8 GB）
2. 文件头是合法的 `GGUF` magic（防止下到 LFS 指针文件 / 半截文件）
3. 把真实文件名写进 `manifest.json` —— `2_run_server.ipynb` 靠它自动定位模型，
   以后换预设只要重跑本 notebook，服务端一个字都不用改


In [ ]:
import json
import struct

ggufs   = sorted(p for p in DOWNLOAD_DIR.rglob("*.gguf") if p.is_file())
models  = [p for p in ggufs if "mmproj" not in p.name.lower()]
mmprojs = [p for p in ggufs if "mmproj" in p.name.lower()]

print("📋 下载结果：\n")
total = 0.0
for p in ggufs:
    gb = p.stat().st_size / 1024**3
    total += gb
    tag = "← 视觉投影" if "mmproj" in p.name.lower() else "← 主模型"
    print(f"  {p.name:<62} {gb:6.2f} GB  {tag}")
print(f"\n  合计: {total:.2f} GB\n")

assert models,  "❌ 没有主模型 gguf —— 检查 PRESET / QUANT，或仓库是否改过文件名。"
assert mmprojs, "❌ 没有 mmproj —— 多模态会失效，但也可能该仓库确实没放视觉投影。"
assert models[0].stat().st_size > 15 * 1024**3, "❌ 主模型文件太小，可能只下到索引或分片。"

# GGUF 文件头校验（magic + version + 张量数）
with open(models[0], "rb") as f:
    head = f.read(24)
magic = head[:4]
assert magic == b"GGUF", f"❌ 不是合法 GGUF（magic={magic!r}），大概率下到了 LFS 指针。"
version, n_tensors, n_kv = struct.unpack("<IQQ", head[4:24])
print(f"🧾 GGUF 头：version={version}  tensors={n_tensors}  kv={n_kv}")

manifest = {
    "source_repo":   REPO_ID,
    "preset":        PRESET,
    "quant":         QUANT,
    "model_file":    models[0].name,
    "mmproj_file":   mmprojs[0].name,
    "model_size_gb": round(models[0].stat().st_size / 1024**3, 2),
}
(DOWNLOAD_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("\n📝 manifest.json 已写入：")
print(json.dumps(manifest, indent=2, ensure_ascii=False))
print("\n✅ 校验通过，可以建 Dataset 了。")


## 📦 Cell 5 — 创建 / 更新 Kaggle Dataset

自动处理两种情况：首次运行 → `datasets create`；已经存在 → `datasets version` 推新版本。

`--dir-mode skip` 是故意的：不压缩直接传。20 GB 的 gguf 再压一遍既费磁盘又费时间，
GGUF 本身已经是压过的二进制。


In [ ]:
import subprocess

# ── 你的 Kaggle 用户名 ────────────────────────────────────────────────────────
KAGGLE_USERNAME = ""   # ← 例如 "johndoe"，去 https://www.kaggle.com/settings 看
# ──────────────────────────────────────────────────────────────────────────────

if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
if not KAGGLE_USERNAME:
    raise ValueError("请先在 Cell 5 顶部填 KAGGLE_USERNAME。")

print(f"👤 用户    : {KAGGLE_USERNAME}")
print(f"📦 Dataset : {KAGGLE_USERNAME}/{DATASET_SLUG}\n")

meta = {
    "title":     DATASET_TITLE,
    "id":        f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
    "licenses":  [{"name": "apache-2.0"}],
    "isPrivate": True,
}
(DOWNLOAD_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

check = subprocess.run(
    ["kaggle", "datasets", "status", f"{KAGGLE_USERNAME}/{DATASET_SLUG}"],
    capture_output=True, text=True,
)

print("⬆️  上传中（20 GB，几分钟到二十分钟，取决于 Kaggle 存储侧）...\n")
if check.returncode == 0:
    print("🔄 Dataset 已存在 → 推送新版本")
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(DOWNLOAD_DIR),
         "-m", f"uncensored preset {PRESET} / {QUANT}", "--dir-mode", "skip"],
        capture_output=True, text=True,
    )
else:
    print("✨ 新建 Dataset")
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(DOWNLOAD_DIR), "--dir-mode", "skip"],
        capture_output=True, text=True,
    )

print(result.stdout)
if result.returncode != 0:
    print("⚠️ stderr:", result.stderr)
    print("   常见原因：KAGGLE_USERNAME 写错 / 该 slug 已被别的账号占用 / 磁盘空间不足。")
else:
    print(f"\n✅ 完成: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_SLUG}")


## ✅ 完成

模型已永久躺在你的 Kaggle 账号里。

下一步：打开 `2_run_server.ipynb`，在 **Input → Add data** 里挂上这个 Dataset：

```
DATASET_NAME = "qwen36-35b-a3b-uncensored-gguf"
```

服务端会读 `/kaggle/input/datasets/<你>/qwen36-35b-a3b-uncensored-gguf/manifest.json`
自己找文件名，所以**不需要再手写 gguf 文件名**。

---

### 想换预设？

改 Cell 2 的 `PRESET`，重跑 Cell 3 → Cell 5。Cell 5 会自动推新版本 Dataset，
`2_run_server.ipynb` 那边一行都不用动。
